# Recommander des films avec ALS sur MovieLens

Ce TP construit un système de recommandation par filtrage collaboratif : à partir des seules notes que des utilisateurs ont données à des films, il prédit les notes qui manquent avec la factorisation de matrice de Spark ML, `ALS`, puis propose à chacun des films qu'il n'a pas encore notés. Vous explorez les notes, mesurez deux références simples, entraînez le modèle et choisissez sa régularisation sur une validation, produisez les recommandations de chaque utilisateur, puis les évaluez comme un classement, contre les films les plus populaires et contre un modèle à préférences implicites.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Spark tourne ici en mode local, sur les deux cœurs de la machine de Colab : un entraînement d'ALS prend une demi-minute, et le notebook entier sept à huit minutes.

## Installation et données

Les données sont [MovieLens 100k](https://grouplens.org/datasets/movielens/100k/), recueillies par le projet GroupLens de l'université du Minnesota sur le site MovieLens, de septembre 1997 à avril 1998 (F. Maxwell Harper et Joseph A. Konstan, « The MovieLens Datasets: History and Context », *ACM Transactions on Interactive Intelligent Systems*, 2015) : 100 000 notes de 1 à 5, données par 943 utilisateurs à 1 682 films, chaque utilisateur en ayant noté au moins 20. Ses conditions d'usage, dans le fichier `README` de l'archive, interdisent de redistribuer les données et de s'en servir à des fins commerciales sans l'accord de GroupLens, et demandent de citer l'article ci-dessus : ce TP les télécharge donc depuis le site de GroupLens.

Deux fichiers de l'archive servent ici, sans ligne d'en-tête. `u.data` a une ligne par note, ses champs séparés par une tabulation : l'identifiant de l'utilisateur, celui du film, la note et sa date, en secondes depuis 1970. `u.item` a une ligne par film, ses champs séparés par `|` : l'identifiant, le titre, puis la date de sortie, l'adresse IMDb et les genres, qui ne servent pas ici. Il est encodé en ISO-8859-1, pas en UTF-8.

In [ ]:
!pip install pyspark
!wget -q -nc https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q -o ml-100k.zip

In [ ]:
import os
import sys
import time

from pyspark.ml.evaluation import RankingEvaluator, RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

La session tourne en local sur deux cœurs, avec quatre partitions pour les agrégations au lieu des 200 par défaut. Les barres de progression de Spark sont masquées : elles encombreraient les sorties. Les processus Python que Spark lance prennent l'interpréteur de ce notebook, `sys.executable` : hors de Colab, Spark prendrait sinon le `python3` du `PATH`, qui peut être d'une autre version de Python que celui du driver, ce que Spark refuse.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
  SparkSession.builder.appName("recommandations-films")
  .master("local[2]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

La table des films ne sert qu'à afficher des titres : elle est lue ici. Sans en-tête, Spark nomme ses colonnes `_c0`, `_c1`… : seuls l'identifiant, renommé `film`, et le titre sont gardés.

In [ ]:
films = spark.read.csv("ml-100k/u.item", sep="|", encoding="iso-8859-1").select(
  F.col("_c0").cast("int").alias("film"), F.col("_c1").alias("titre")
)
films.show(3, truncate=False)

## 1. Les notes

Lisez `ml-100k/u.data` dans `notes`, avec le schéma `"utilisateur INT, film INT, note INT, horodatage LONG"`, et mettez-le en cache. Combien y a-t-il de notes, d'utilisateurs et de films ? Quelle part de la matrice utilisateurs × films est remplie ?

Affichez la répartition des notes et la note moyenne, puis les dix films les plus notés, avec leur titre, leur nombre de notes et leur note moyenne.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`spark.read.csv` prend le séparateur (`sep="\t"`, la tabulation) et le schéma (`schema=`), une chaîne de noms et de types, qui évite à Spark de deviner les types. Le nombre d'utilisateurs est celui des valeurs distinctes de `utilisateur`. La part remplie est le nombre de notes divisé par le nombre de cases de la matrice, utilisateurs × films.

```python
notes = spark.read.csv("ml-100k/u.data", sep=..., schema=...).cache()
nb_notes = notes.count()
nb_utilisateurs = notes.select("utilisateur").distinct().count()
nb_films = ...
print(nb_notes, nb_utilisateurs, nb_films, nb_notes / (...))
notes.groupBy("note").count().orderBy("note").show()
notes.agg(F.mean("note")).show()
(
  notes.groupBy("film")
  .agg(F.count("*").alias("nb_notes"), F.mean("note").alias("moyenne"))
  .join(films, ...)
  .orderBy(F.desc(...))
  .show(10, truncate=False)
)
```

### Solution

In [ ]:
notes = spark.read.csv(
  "ml-100k/u.data",
  sep="\t",
  schema="utilisateur INT, film INT, note INT, horodatage LONG",
).cache()
nb_notes = notes.count()
nb_utilisateurs = notes.select("utilisateur").distinct().count()
nb_films = notes.select("film").distinct().count()
print(nb_notes, nb_utilisateurs, nb_films, nb_notes / (nb_utilisateurs * nb_films))
notes.groupBy("note").count().orderBy("note").show()
notes.agg(F.mean("note")).show()
(
  notes.groupBy("film")
  .agg(F.count("*").alias("nb_notes"), F.mean("note").alias("moyenne"))
  .join(films, "film")
  .orderBy(F.desc("nb_notes"))
  .show(10, truncate=False)
)
# 100 000 notes, 943 utilisateurs et 1 682 films : 6,3 % de la matrice est remplie,
# le reste est ce qu'il faut prédire. Les notes vont surtout de 3 à 5 (4 est la plus
# fréquente), pour une moyenne de 3,53. Star Wars est le film le plus noté (583 notes,
# moyenne 4,36) ; les suivants sont surtout des succès de 1996 et 1997, l'époque de
# la collecte.

## 2. Découper et deux références

Coupez `notes` en une partie entraînement (80 %) et une partie test (20 %) avec `randomSplit` et la graine 42, dans `entrainement` et `test`, en cache. Créez l'évaluateur `rmse`, un `RegressionEvaluator` qui compare la colonne `note` à la colonne `prediction` (sa mesure par défaut est la RMSE, la racine de l'erreur quadratique moyenne).

Mesurez ensuite la RMSE sur le test de deux prédictions simples, les références qu'un modèle doit battre : la note moyenne de l'entraînement, la même pour toutes les notes, puis la note moyenne du film dans l'entraînement. Combien de notes du test portent sur un film absent de l'entraînement ? Que deviennent-elles dans la seconde référence ?


**Documentation utile**

- [pyspark.sql.functions.lit](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.lit.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`randomSplit` rend une liste de DataFrames, une par proportion. `F.lit(valeur)` crée une colonne constante. Pour la seconde référence, calculez la moyenne de chaque film dans une colonne `prediction` et joignez-la au test sur `film` : la jointure ordinaire (`"inner"`) laisse de côté les notes de films absents de l'entraînement, que la jointure `"left_anti"` compte.

```python
entrainement, test = notes.randomSplit([..., ...], seed=42)
entrainement.cache()
test.cache()
print(entrainement.count(), test.count())
rmse = RegressionEvaluator(labelCol=..., predictionCol=...)
moyenne = entrainement.agg(F.mean("note")).first()[0]
print(rmse.evaluate(test.withColumn("prediction", F.lit(...))))
moyennes_films = entrainement.groupBy("film").agg(...)
print(rmse.evaluate(test.join(moyennes_films, "film")))
print(test.join(moyennes_films, "film", ...).count())
```

### Solution

In [ ]:
entrainement, test = notes.randomSplit([0.8, 0.2], seed=42)
entrainement.cache()
test.cache()
print(entrainement.count(), test.count())
rmse = RegressionEvaluator(labelCol="note", predictionCol="prediction")
moyenne = entrainement.agg(F.mean("note")).first()[0]
print(rmse.evaluate(test.withColumn("prediction", F.lit(moyenne))))
moyennes_films = entrainement.groupBy("film").agg(F.mean("note").alias("prediction"))
print(rmse.evaluate(test.join(moyennes_films, "film")))
print(test.join(moyennes_films, "film", "left_anti").count())
# 79 901 notes d'entraînement et 20 099 de test. Prédire la moyenne (3,53) donne une
# RMSE de 1,128 ; prédire la moyenne du film, 1,020. 46 notes du test portent sur
# des films absents de l'entraînement : la jointure les laisse de côté, et la seconde
# référence n'est mesurée que sur les 20 053 autres.

## 3. Entraîner ALS

Entraînez sur `entrainement` un modèle `ALS`, dans `modele` : utilisateurs dans `utilisateur`, films dans `film`, notes dans `note`, rang 10, `regParam` 0,1, 10 itérations et la graine 42. Mesurez la durée de l'entraînement. Calculez la RMSE du modèle sur le test : que se passe-t-il, et pourquoi ?

Réglez la stratégie de démarrage à froid du modèle (*cold start*) à `"drop"`, puis recalculez sa RMSE sur le test, et sur l'entraînement. Comparez aux références de l'exercice 2.

Le modèle a appris un vecteur de facteurs par utilisateur, `modele.userFactors`, et un par film, `modele.itemFactors`. Combien chacune de ces tables a-t-elle de lignes ? Affichez le vecteur d'un film. Combien de nombres le modèle a-t-il appris, pour combien de notes ?


**Documentation utile**

- [pyspark.ml.recommendation.ALSModel](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.recommendation.ALSModel.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`ALS` prend les noms des colonnes (`userCol`, `itemCol`, `ratingCol`) et les hyperparamètres `rank`, `regParam`, `maxIter` et `seed`. Une note du test dont l'utilisateur ou le film n'a pas de vecteur reçoit la prédiction `NaN`, et une moyenne qui contient un `NaN` vaut `NaN`. `modele.setColdStartStrategy("drop")` règle le modèle déjà entraîné : pas besoin de le réentraîner.

```python
debut = time.time()
modele = ALS(
  userCol=...,
  itemCol=...,
  ratingCol=...,
  rank=...,
  regParam=...,
  maxIter=...,
  seed=42,
).fit(entrainement)
print(time.time() - debut)
print(rmse.evaluate(modele.transform(test)))
modele.setColdStartStrategy(...)
print(rmse.evaluate(modele.transform(test)))
print(rmse.evaluate(modele.transform(...)))
print(modele.userFactors.count(), modele.itemFactors.count())
modele.itemFactors.show(1, truncate=False)
```

### Solution

In [ ]:
debut = time.time()
modele = ALS(
  userCol="utilisateur",
  itemCol="film",
  ratingCol="note",
  rank=10,
  regParam=0.1,
  maxIter=10,
  seed=42,
).fit(entrainement)
print(time.time() - debut)
print(rmse.evaluate(modele.transform(test)))
modele.setColdStartStrategy("drop")
print(rmse.evaluate(modele.transform(test)))
print(rmse.evaluate(modele.transform(entrainement)))
print(modele.userFactors.count(), modele.itemFactors.count())
modele.itemFactors.show(1, truncate=False)
# L'entraînement prend une demi-minute, selon la machine. Sans stratégie de
# démarrage à froid, la RMSE vaut NaN : les 46 notes de films absents de
# l'entraînement n'ont pas de vecteur de film, donc pas de prédiction. Avec "drop",
# elles sont retirées : RMSE de 0,920 sur le test, contre 1,020 pour la moyenne du
# film, et de 0,747 sur l'entraînement. 943 vecteurs d'utilisateurs et 1 647 de films
# (ceux de l'entraînement), de 10 nombres chacun : (943 + 1 647) × 10 = 25 900
# paramètres pour 79 901 notes.

## 4. Choisir la régularisation

Choisissez `regParam` parmi 0,01, 0,05, 0,1 et 0,3 sans regarder le test. `TrainValidationSplit` coupe `entrainement` en une partie d'apprentissage (80 %) et une partie de validation, entraîne un modèle par valeur sur la première, le mesure sur la seconde, puis réentraîne le meilleur sur tout `entrainement`.

Donnez-lui un `ALS` aux paramètres de l'exercice 3, sans `regParam` et avec `coldStartStrategy="drop"`, une grille des quatre valeurs construite par `ParamGridBuilder`, l'évaluateur `rmse` et la graine 42. Ajustez-le sur `entrainement`, dans `modele_reglage`.

Affichez la RMSE de validation de chaque valeur. Laquelle surapprend ? Laquelle sous-apprend ? Quelle valeur est retenue ? Mesurez enfin, une seule fois, la RMSE du modèle retenu sur le test, et comparez-la à celle de l'exercice 3.

Le test ne sert ainsi qu'au modèle retenu : comparer des modèles sur le test et garder le meilleur, ce serait le choisir sur le test, dont la mesure deviendrait optimiste. `CrossValidator` fait de même sur plusieurs découpages, pour un choix plus sûr et un coût multiplié par le nombre de plis.


**Documentation utile**

- [pyspark.ml.tuning.TrainValidationSplit](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.TrainValidationSplit.html)
- [pyspark.ml.tuning.TrainValidationSplitModel](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.TrainValidationSplitModel.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`ParamGridBuilder().addGrid(als.regParam, [...]).build()` rend la liste des combinaisons à essayer, ici quatre. `TrainValidationSplit` prend l'estimateur (`estimator`), la grille (`estimatorParamMaps`), l'évaluateur (`evaluator`), la part d'apprentissage (`trainRatio=0.8`) et `seed`. Son modèle garde les RMSE de validation dans l'ordre de la grille (`validationMetrics`) ; `bestModel` est le meilleur, réentraîné sur tout `entrainement`, avec `coldStartStrategy="drop"` comme l'estimateur. Comptez cinq entraînements : un par valeur, puis le meilleur.

```python
als = ALS(
  userCol="utilisateur",
  itemCol="film",
  ratingCol="note",
  rank=10,
  maxIter=10,
  coldStartStrategy=...,
  seed=42,
)
grille = ParamGridBuilder().addGrid(als.regParam, [...]).build()
reglage = TrainValidationSplit(
  estimator=als,
  estimatorParamMaps=...,
  evaluator=...,
  trainRatio=...,
  seed=42,
)
modele_reglage = reglage.fit(entrainement)
for parametres, validation in zip(grille, modele_reglage.validationMetrics):
  print(parametres[als.regParam], f"{validation:.3f}")
print(f"{rmse.evaluate(modele_reglage.bestModel.transform(...)):.3f}")
```

### Solution

In [ ]:
als = ALS(
  userCol="utilisateur",
  itemCol="film",
  ratingCol="note",
  rank=10,
  maxIter=10,
  coldStartStrategy="drop",
  seed=42,
)
grille = ParamGridBuilder().addGrid(als.regParam, [0.01, 0.05, 0.1, 0.3]).build()
reglage = TrainValidationSplit(
  estimator=als,
  estimatorParamMaps=grille,
  evaluator=rmse,
  trainRatio=0.8,
  seed=42,
)
modele_reglage = reglage.fit(entrainement)
for parametres, validation in zip(grille, modele_reglage.validationMetrics):
  print(parametres[als.regParam], f"{validation:.3f}")
print(f"{rmse.evaluate(modele_reglage.bestModel.transform(test)):.3f}")
# RMSE de validation : 1,164 pour regParam 0,01, 0,993 pour 0,05, 0,944 pour 0,1 et
# 0,987 pour 0,3 : TrainValidationSplit retient 0,1. À 0,01, le modèle surapprend :
# sa validation est pire que la moyenne du film (1,020) ; l'exercice 3 montrait déjà,
# à 0,1, l'écart entre entraînement (0,747) et test (0,920). À 0,3, il sous-apprend :
# trop contraint, il remonte à 0,987. Le modèle retenu, réentraîné sur tout
# entrainement, est celui de l'exercice 3 : 0,920 sur le test, mesuré une seule fois.
# Sa validation était plus haute (0,944) : chaque modèle de la grille n'avait appris
# que sur 80 % de l'entraînement.

## 5. Recommander

`modele.recommendForAllUsers(n)` rend, pour chaque utilisateur, les `n` films de plus forte note prédite, dans une colonne `recommendations` : une liste de couples (`film`, `rating`). Ces films comprennent ceux que l'utilisateur a déjà notés, qu'il ne faut pas lui proposer.

La cellule suivante reprend la recette de la frame « Recommandations en production ». `dix_meilleurs(scores)` prend un DataFrame de colonnes `utilisateur`, `film` et `score`, en retire les couples (utilisateur, film) déjà présents dans `entrainement`, et garde pour chaque utilisateur les dix films de plus fort score, numérotés de 1 à 10 dans la colonne `rang`. `recommander(modele)` applique `dix_meilleurs` aux 200 premières recommandations d'un modèle ALS, une ligne par couple. La cellule applique `recommander` à `modele` (régularisation 0,1), dans `recommandations_als`, compte ses lignes, affiche les films proposés à l'utilisateur 1, avec leur titre et leur nombre de notes dans l'entraînement (`nb_notes_films`), et enregistre `recommandations_als` en Parquet : c'est ce qu'écrirait un job de recommandation par lots, relancé chaque nuit. Exécutez-la.

In [ ]:
nb_notes_films = entrainement.groupBy("film").agg(F.count("*").alias("nb_notes"))


def dix_meilleurs(scores):
  ordre = Window.partitionBy("utilisateur").orderBy(F.desc("score"))
  return (
    scores.join(entrainement, ["utilisateur", "film"], "left_anti")
    .withColumn("rang", F.row_number().over(ordre))
    .filter("rang <= 10")
  )


def recommander(modele):
  return dix_meilleurs(
    modele.recommendForAllUsers(200)
    .select("utilisateur", F.explode("recommendations").alias("r"))
    .select("utilisateur", "r.film", F.col("r.rating").alias("score"))
  )


recommandations_als = recommander(modele).cache()
print(recommandations_als.count())
(
  recommandations_als.filter("utilisateur = 1")
  .join(films, "film")
  .join(nb_notes_films, "film")
  .orderBy("rang")
  .show(truncate=False)
)
recommandations_als.write.mode("overwrite").parquet("recommandations.parquet")

Combien de notes ont, dans l'entraînement, les films proposés à l'utilisateur 1 ? Que vaut la note prédite d'un film que presque personne n'a noté ?

Ajoutez à `recommander` un paramètre `min_notes`, 0 par défaut, qui écarte des candidats les films de moins de `min_notes` notes dans l'entraînement, avant de garder les dix meilleurs. Appliquez-la à `modele` avec `min_notes=20`, dans `recommandations_filtrees`, en cache : chaque utilisateur a-t-il encore ses dix films ? Quels films l'utilisateur 1 reçoit-il ? L'exercice 6 mesure ce que le filtre change à la qualité des recommandations.

In [ ]:
def recommander(modele, min_notes=0): ...


# Votre code ici

### Conseils supplémentaires

Le nombre de notes de chaque film dans l'entraînement est dans `nb_notes_films` : gardez-en les films d'au moins `min_notes` notes et joignez-les aux candidats sur `film` (jointure ordinaire), avant `dix_meilleurs`. Avec `min_notes=0`, aucun film n'est écarté : `recommander(modele)` rend les mêmes recommandations qu'avant. Les 200 candidats suffisent encore à laisser dix films à chacun avec un filtre à 20 notes : comptez les lignes de `recommandations_filtrees`.

```python
def recommander(modele, min_notes=0):
  frequents = nb_notes_films.filter(F.col("nb_notes") >= ...).select("film")
  return dix_meilleurs(
    modele.recommendForAllUsers(200)
    .select("utilisateur", F.explode("recommendations").alias("r"))
    .select("utilisateur", "r.film", F.col("r.rating").alias("score"))
    .join(..., "film")
  )


recommandations_filtrees = recommander(modele, min_notes=...).cache()
print(recommandations_filtrees.count())
(
  recommandations_filtrees.filter(...)
  .join(films, "film")
  .join(nb_notes_films, "film")
  .orderBy("rang")
  .show(truncate=False)
)
```

### Solution

In [ ]:
def recommander(modele, min_notes=0):
  frequents = nb_notes_films.filter(F.col("nb_notes") >= min_notes).select("film")
  return dix_meilleurs(
    modele.recommendForAllUsers(200)
    .select("utilisateur", F.explode("recommendations").alias("r"))
    .select("utilisateur", "r.film", F.col("r.rating").alias("score"))
    .join(frequents, "film")
  )


recommandations_filtrees = recommander(modele, min_notes=20).cache()
print(recommandations_filtrees.count())
(
  recommandations_filtrees.filter("utilisateur = 1")
  .join(films, "film")
  .join(nb_notes_films, "film")
  .orderBy("rang")
  .show(truncate=False)
)
# 9 430 lignes dans recommandations_als : dix films pour chacun des 943
# utilisateurs. À l'utilisateur 1, le modèle propose des films très bien notés, mais
# aussi des films que presque personne n'a vus : Maya Lin: A Strong Clear Vision,
# noté une seule fois (un 5), Angel Baby (3 notes), Faust (4), Pather Panchali (6).
# Le vecteur d'un film peu noté est peu contraint, et sa note prédite peut monter
# haut. Avec min_notes=20, encore 9 430 lignes : 200 candidats suffisent. Les films
# de l'utilisateur 1 ont désormais de 26 à 473 notes : The Wrong Trousers, A Close
# Shave, L.A. Confidential, Star Wars, Casablanca…

## 6. Évaluer le classement

La RMSE mesure les notes prédites, pas les dix films proposés. On dit pertinent un film du test que l'utilisateur a noté 4 ou 5 : la précision à 10 est la part de films pertinents parmi les dix recommandés, en moyenne sur les utilisateurs.

Construisez `pertinents` : pour chaque utilisateur qui a au moins un film pertinent dans le test, la liste de ses films pertinents, en nombres à virgule (`double`, le type qu'attend `RankingEvaluator`), en cache. Écrivez une fonction `precision_a_10(recommandations)` qui rassemble les dix films de chaque utilisateur d'une table comme `recommandations_als` en une liste ordonnée par rang, la joint à `pertinents` et rend la précision à 10 que calcule `RankingEvaluator`. Calculez-la pour :

- `recommandations_als` ;
- les recommandations de `modele` qui écartent les films de moins de 20, puis de moins de 50 notes (`recommander(modele, min_notes=...)`, exercice 5) ;
- la popularité, dans `recommandations_populaires` : à chaque utilisateur, les dix films les plus notés de l'entraînement qu'il n'a pas encore notés ;
- un modèle ALS à préférences implicites, dans `modele_implicite` : les paramètres de l'exercice 3, plus `implicitPrefs=True`. La note n'y est plus une valeur à prédire, mais la confiance dans le fait que l'utilisateur a choisi de voir le film.

Qu'en concluez-vous ? Affichez les recommandations du modèle implicite à l'utilisateur 1, avec leur nombre de notes, et comparez-les à celles de l'exercice 5.


**Documentation utile**

- [pyspark.sql.functions.collect_list](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.collect_list.html)
- [pyspark.sql.functions.sort_array](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sort_array.html)
- [pyspark.sql.DataFrame.crossJoin](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.crossJoin.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`F.collect_list` rassemble les valeurs d'un groupe en une liste, dans un ordre quelconque : rassemblez des couples `F.struct("rang", film)`, triez la liste avec `F.sort_array`, qui trie selon le premier champ, puis gardez le champ `film` de chaque couple (`F.col("liste.film")`). `RankingEvaluator` compare la colonne `predictionCol` (les films recommandés, dans l'ordre) à la colonne `labelCol` (les pertinents), avec `metricName="precisionAtK"` et `k=10`. Pour la popularité, `crossJoin` associe chaque utilisateur à chaque film, avec son nombre de notes comme score.

```python
pertinents = (
  test.filter(...)
  .groupBy("utilisateur")
  .agg(F.collect_list(F.col("film").cast("double")).alias("pertinents"))
  .cache()
)
precision = RankingEvaluator(
  predictionCol="recommandes", labelCol=..., metricName=..., k=...
)


def precision_a_10(recommandations):
  listes = recommandations.groupBy("utilisateur").agg(
    F.sort_array(
      F.collect_list(F.struct("rang", F.col("film").cast("double").alias("film")))
    ).alias("liste")
  )
  recommandes = listes.select("utilisateur", F.col(...).alias("recommandes"))
  return precision.evaluate(recommandes.join(pertinents, "utilisateur"))


print(precision_a_10(recommandations_als))
for min_notes in [20, 50]:
  print(min_notes, precision_a_10(recommander(modele, min_notes=...)))
utilisateurs = notes.select("utilisateur").distinct()
popularite = entrainement.groupBy("film").agg(F.count("*").alias("score"))
recommandations_populaires = dix_meilleurs(utilisateurs.crossJoin(...))
print(precision_a_10(recommandations_populaires))
modele_implicite = ALS(..., implicitPrefs=True, coldStartStrategy="drop").fit(
  entrainement
)
recommandations_implicites = recommander(...)
print(precision_a_10(recommandations_implicites))
(
  recommandations_implicites.filter("utilisateur = 1")
  .join(films, "film")
  .join(nb_notes_films, "film")
  .orderBy("rang")
  .show(truncate=False)
)
```

### Solution

In [ ]:
pertinents = (
  test.filter("note >= 4")
  .groupBy("utilisateur")
  .agg(F.collect_list(F.col("film").cast("double")).alias("pertinents"))
  .cache()
)
precision = RankingEvaluator(
  predictionCol="recommandes", labelCol="pertinents", metricName="precisionAtK", k=10
)


def precision_a_10(recommandations):
  listes = recommandations.groupBy("utilisateur").agg(
    F.sort_array(
      F.collect_list(F.struct("rang", F.col("film").cast("double").alias("film")))
    ).alias("liste")
  )
  recommandes = listes.select("utilisateur", F.col("liste.film").alias("recommandes"))
  return precision.evaluate(recommandes.join(pertinents, "utilisateur"))


print(precision_a_10(recommandations_als))
for min_notes in [20, 50]:
  print(min_notes, precision_a_10(recommander(modele, min_notes=min_notes)))
utilisateurs = notes.select("utilisateur").distinct()
popularite = entrainement.groupBy("film").agg(F.count("*").alias("score"))
recommandations_populaires = dix_meilleurs(utilisateurs.crossJoin(popularite))
print(precision_a_10(recommandations_populaires))
modele_implicite = ALS(
  userCol="utilisateur",
  itemCol="film",
  ratingCol="note",
  rank=10,
  regParam=0.1,
  maxIter=10,
  implicitPrefs=True,
  coldStartStrategy="drop",
  seed=42,
).fit(entrainement)
recommandations_implicites = recommander(modele_implicite)
print(precision_a_10(recommandations_implicites))
(
  recommandations_implicites.filter("utilisateur = 1")
  .join(films, "film")
  .join(nb_notes_films, "film")
  .orderBy("rang")
  .show(truncate=False)
)
# Précision à 10 : 0,035 pour ALS, 0,080 et 0,097 en écartant les films de moins de
# 20 et de 50 notes, 0,132 pour la popularité, 0,250 pour le modèle implicite. Le
# modèle de la meilleure RMSE fait bien moins que la popularité : il met en tête des
# films que presque personne n'a vus. Or, hors ligne, un film n'est pertinent que si
# l'utilisateur l'a vu et noté : recommander un film rare ne rapporte guère de point,
# même s'il lui plairait (seul un test A/B le dirait). Écarter les films rares aide,
# sans rattraper la popularité : parmi les films vus par beaucoup, le modèle classe
# encore selon la note qu'il prédit, pas selon la chance que l'utilisateur ait vu le
# film. Le modèle
# implicite apprend ce que chacun choisit de voir : ses dix films pour l'utilisateur 1
# ont tous plus de 100 notes. La précision à 10 divise toujours par 10, même pour un
# utilisateur qui n'a que deux films pertinents dans le test : aucun modèle
# n'approcherait 1.

## Fin

Arrêtez la session, qui libère les exécuteurs.

In [ ]:
spark.stop()